# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *(35)* |
| **Integrantes** | *(Paula Andrea Celis Cano)* |
| **Caso de estudio** | *(Wanderbricks u otro)* |
| **Fecha de entrega** | Miercoles 26 de agosto |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.*
Wanderbricks es una plataforma simulada de reservas de alojamientos que contiene información sobre usuarios, propiedades, reservas, pagos y reseñas. También registra las actividades que realizan los usuarios mientras navegan.
El objetivo es organizar y analizar estos datos para conocer qué propiedades y destinos tienen mayor demanda, cuáles generan más ingresos, qué tan satisfechos están los huéspedes y cómo la navegación de los usuarios se relaciona con las reservas.


In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:
# Cantidad de registros por tabla

tablas = [
    "amenities",
    "booking_updates",
    "bookings",
    "clickstream",
    "countries",
    "customer_support_logs",
    "destinations",
    "employees",
    "hosts",
    "page_views",
    "payments",
    "properties",
    "property_amenities",
    "property_images",
    "reviews",
    "users"
]

resultados = []

for tabla in tablas:
   cantidad = spark.table(f"samples.wanderbricks.{tabla}").count()
   resultados.append((tabla, cantidad))

display(
spark.createDataFrame(
    resultados,
    ["tabla", "registros"]
    ).orderBy("registros", ascending=False)
)

In [0]:
# Estructura de las principales tablas del caso

tablas_principales = [
    "users",
    "hosts",
    "properties",
    "destinations",
    "bookings",
    "payments",
    "reviews",
    "clickstream",
    "page_views"
]

for tabla in tablas_principales:
    print(f"\n==== {tabla.upper()} ====")
    spark.table(f"samples.wanderbricks.{tabla}").printSchema()

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

### Volumen de los datos

La exploración inicial identificó 16 tablas en el catálogo `samples.wanderbricks`. El volumen de información es variable entre las diferentes entidades del sistema.

Entre las tablas con mayor cantidad de registros se encuentran `page_views`, con 500.000 registros; `users`, con 124.509; `property_amenities`, con 118.108; `clickstream`, con 100.000; y `reviews`, con 99.793 registros.

También se identificaron tablas de menor tamaño, como `destinations`, con 42 registros, y `countries`, con 168 registros. Esta diferencia de volumen muestra que el conjunto combina tablas transaccionales y de eventos con tablas dimensionales de menor tamaño.

El volumen y la variedad observados son relevantes para la selección de la arquitectura, ya que el sistema debe permitir trabajar tanto con información estructurada de las operaciones de la plataforma como con grandes cantidades de eventos de navegación.

### Variedad y tipos de datos

El dataset Wanderbricks está compuesto por 16 tablas que representan diferentes componentes de una plataforma de reservas de alojamientos. En la exploración se analizaron las tablas principales relacionadas con usuarios, anfitriones, propiedades, destinos, reservas, pagos, reseñas y navegación.

Los datos presentan diferentes tipos de información. Se encuentran identificadores numéricos de tipo `long`, textos de tipo `string`, fechas de tipo `date`, marcas de tiempo de tipo `timestamp`, valores numéricos de tipo `integer`, `float` y `double`, valores monetarios de tipo `decimal` y campos booleanos.

Las tablas `users`, `hosts`, `properties`, `destinations`, `bookings`, `payments` y `reviews` presentan principalmente estructuras tabulares y relaciones mediante identificadores. Por ejemplo, `bookings` contiene `user_id` y `property_id`, mientras que `payments` contiene `booking_id`, permitiendo relacionar las reservas con los pagos.

La tabla `clickstream` presenta una característica semiestructurada importante: el campo `metadata` es de tipo `struct` y contiene los atributos `device` y `referrer`. Esto permite analizar eventos de navegación junto con información adicional del dispositivo y del origen de la visita sin tener que convertir inicialmente toda la estructura en columnas independientes.

La tabla `page_views` registra las visitas a páginas e incluye información como dispositivo, URL, referente, usuario, propiedad y fecha/hora. Las tablas `properties` y `destinations` también permiten trabajar una dimensión geográfica mediante identificadores, países, estados o provincias y coordenadas de las propiedades.

In [0]:
# Relaciones principales identificadas en el modelo

relaciones = [
    ("bookings", "users", "bookings.user_id = users.user_id"),
    ("bookings", "properties", "bookings.property_id = properties.property_id"),
    ("payments", "bookings", "payments.booking_id = bookings.booking_id"),
    ("reviews", "bookings", "reviews.booking_id = bookings.booking_id"),
    ("reviews", "users", "reviews.user_id = users.user_id"),
    ("reviews", "properties", "reviews.property_id = properties.property_id"),
    ("properties", "hosts", "properties.host_id = hosts.host_id"),
    ("properties", "destinations", "properties.destination_id = destinations.destination_id"),
    ("clickstream", "users", "clickstream.user_id = users.user_id"),
    ("clickstream", "properties", "clickstream.property_id = properties.property_id"),
    ("page_views", "users", "page_views.user_id = users.user_id"),
    ("page_views", "properties", "page_views.property_id = properties.property_id")
]

display(
    spark.createDataFrame(
        relaciones,
        ["tabla_origen", "tabla_relacionada", "condicion"]
    )
)

In [0]:
from pyspark.sql import functions as F

tablas_calidad = [
    "users",
    "hosts",
    "properties",
    "destinations",
    "bookings",
    "payments",
    "reviews",
    "clickstream",
    "page_views"
]

for tabla in tablas_calidad:
    df = spark.table(f"samples.wanderbricks.{tabla}")
    
    nulos = df.select(
        *[
            F.sum(F.col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ]
    ).collect()[0].asDict()
    
    total_nulos = sum(v or 0 for v in nulos.values())
    
    print(f"{tabla}: {total_nulos} valores nulos")

### Calidad de los datos

Como parte de la exploración se realizó una revisión de valores nulos en las principales tablas del dataset.

La tabla `users` presentó 62.091 valores nulos sobre un total de 124.509 registros, lo que representa aproximadamente el 49,9 % de los registros. Este resultado indica que existen campos con información faltante que deberán ser analizados antes de utilizarlos en procesos analíticos.

La tabla `reviews` presentó 952 valores nulos sobre 99.793 registros, aproximadamente el 1,0 % de sus registros. En comparación con `users`, este porcentaje es considerablemente menor.

Las tablas `hosts`, `properties`, `destinations`, `bookings`, `payments`, `clickstream` y `page_views` no presentaron valores nulos en la revisión realizada.

Estos resultados muestran que la calidad de los datos no es uniforme entre las diferentes entidades. Por esta razón, en la capa Silver será necesario definir reglas de limpieza y tratamiento de valores faltantes, especialmente para los datos de usuarios y reseñas.

In [0]:
from pyspark.sql import functions as F

tablas = ["users", "reviews"]

for tabla in tablas:
    print(f"\n========== {tabla.upper()} ==========")
    
    df = spark.table(f"samples.wanderbricks.{tabla}")
    
    resultado = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    
    display(resultado)

### Análisis de calidad de los datos

La revisión de valores nulos permitió identificar que la mayor cantidad de información faltante se concentra en las tablas `users` y `reviews`.

En la tabla `users` se encontraron 62.091 valores nulos. De estos, 61.967 corresponden a la columna `company_name` y 124 a la columna `email`. La ausencia de `company_name` puede estar relacionada con usuarios que no corresponden a cuentas empresariales, por lo que no necesariamente representa un error en los datos. En cambio, los registros sin correo electrónico deben considerarse al realizar análisis que dependan de este atributo.

En la tabla `reviews` se identificaron 952 valores nulos: 476 corresponden a `comment` y 476 a `rating`. Esto indica que existen reseñas que no contienen comentario, valoración o ambos atributos. Para los análisis de satisfacción será necesario establecer reglas que permitan determinar cómo tratar estos registros.

Las demás tablas revisadas (`hosts`, `properties`, `destinations`, `bookings`, `payments`, `clickstream` y `page_views`) no presentaron valores nulos en esta revisión.

Estos resultados muestran que la calidad de los datos debe ser considerada durante la construcción de la capa Silver. No todos los valores nulos deben eliminarse automáticamente; su tratamiento debe depender del significado de cada atributo y de la pregunta analítica que se quiera responder.

In [0]:
# TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*

| Criterio                             | Relacional                                          | NoSQL                                    | Lakehouse                              | Decisión para Wanderbricks |
| ------------------------------------ | --------------------------------------------------- | ---------------------------------------- | -------------------------------------  | -------------------------- |
| Datos estructurados                  | Muy adecuado para usuarios, reservas y pagos        | Adecuado                                 | Muy adecuado                               | Lakehouse                  |
| Relaciones entre tablas              | Muy adecuado                                        | Puede requerir duplicación o referencias | Muy adecuado                               | Lakehouse                  |
| `clickstream.metadata` como `struct` | Requiere mayor rigidez                              | Muy flexible                             | Flexible con Spark/Delta              | **Lakehouse**     |
| Texto de reseñas                     | Adecuado                                            | Adecuado                                 | Muy adecuado para integrar y analizar | **Lakehouse**    |
| Grandes volúmenes de eventos         | Adecuado, pero puede requerir mayor infraestructura | Adecuado                                 | Muy adecuado                               | **Lakehouse**              |
| Evolución de esquema                 | Más rígida                                          | Flexible                                 | Flexible mediante Delta               | **Lakehouse**    |
| Análisis histórico                   | Adecuado                                            | Variable                                 | Muy adecuado con Delta/Time Travel    | **Lakehouse**         |




**Referencias (APA 7):**

### Justificación de la decisión

Se selecciona el modelo Lakehouse para Wanderbricks porque el dataset combina información estructurada y semiestructurada que debe ser analizada de manera conjunta. El conjunto de datos contiene 16 tablas relacionadas, incluyendo usuarios, anfitriones, propiedades, reservas, pagos y reseñas, además de información de navegación mediante `clickstream` y `page_views`.

El modelo relacional sería adecuado para las entidades estructuradas y para las relaciones entre tablas, como `bookings` con `users` y `properties`, o `payments` con `bookings`. Sin embargo, presenta una mayor rigidez frente a estructuras semiestructuradas como `clickstream.metadata`, que es un `struct` con los campos `device` y `referrer`.

Un modelo NoSQL, especialmente documental, tendría ventajas para manejar estructuras flexibles como el `metadata` del clickstream y podría adaptarse a cambios en el esquema. Sin embargo, Wanderbricks requiere realizar análisis que relacionan diferentes entidades, por ejemplo, usuarios, propiedades, reservas, pagos, destinos y reseñas. Estas consultas pueden involucrar varias tablas y agregaciones, por lo que un modelo orientado exclusivamente a documentos no resulta tan conveniente para todo el caso.

El Lakehouse permite combinar las ventajas necesarias para este dataset. Puede manejar datos estructurados y semiestructurados mediante Spark, conservar estructuras anidadas como `struct`, y utilizar Delta Lake para implementar las capas Bronze y Silver, controlar la evolución del esquema y realizar consultas sobre diferentes versiones de los datos. Además, permite trabajar con los volúmenes observados en el dataset, como los 500.000 registros de `page_views`, más de 124.000 usuarios y 100.000 eventos de `clickstream`.

Por estas características, el Lakehouse es la alternativa que mejor se adapta a Wanderbricks, ya que permite integrar las diferentes fuentes de información y preparar los datos para consultas analíticas sobre demanda, comportamiento de los usuarios, ingresos y satisfacción de los huéspedes.

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
# Configuración del catálogo, esquema y volumen

CATALOGO = "bigdata_grupo35"
ESQUEMA = "wanderbricks"
VOLUMEN = "datos_crudos"

# Crear catálogo
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")

# Crear esquema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")

# Crear volumen para almacenar los datos crudos
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

# Seleccionar el catálogo y esquema de trabajo
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.1 Catálogo, esquema y volumen

Para la implementación se creó un catálogo propio para el grupo, un esquema denominado `wanderbricks` y un volumen denominado `datos_crudos`.

El catálogo permite organizar los objetos utilizados durante el proyecto, mientras que el esquema agrupa las tablas relacionadas con el caso de estudio. El volumen se utilizará como espacio para almacenar los datos crudos antes de su procesamiento en las capas del Lakehouse.

La fuente original utilizada es `samples.wanderbricks`, proporcionada por Databricks. Los datos originales no serán modificados directamente; se utilizarán como fuente para construir las capas Bronze y Silver del proyecto.

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# TODO: ingerir las tablas del caso a la capa bronce, con esquema explícito
# Verificar el catálogo y esquema de trabajo

print("Catálogo actual:", spark.sql("SELECT current_catalog()").collect()[0][0])
print("Esquema actual:", spark.sql("SELECT current_schema()").collect()[0][0])

In [0]:
tablas_bronze = [
    "users",
    "hosts",
    "properties",
    "destinations",
    "bookings",
    "payments",
    "reviews",
    "clickstream",
    "page_views"
]

for tabla in tablas_bronze:
    df = spark.table(f"samples.wanderbricks.{tabla}")
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bronze_{tabla}")
    )
    
    print(f"Tabla Bronze creada: bronze_{tabla}")


In [0]:
%sql
SHOW TABLES IN bigdata_grupo35.wanderbricks;

In [0]:
%sql
DESCRIBE DETAIL bigdata_grupo35.wanderbricks.bronze_bookings;

In [0]:
%sql
SHOW TABLES IN bigdata_grupo35.wanderbricks;

_### Resultado de la ingesta Bronze

La consulta `SHOW TABLES` confirma la creación de nueve tablas persistentes en el esquema `bigdata_grupo35.wanderbricks`. Estas tablas corresponden a las principales entidades utilizadas en el análisis de Wanderbricks: usuarios, anfitriones, propiedades, destinos, reservas, pagos, reseñas y eventos de navegación.

Las tablas fueron creadas a partir de las fuentes originales de `samples.wanderbricks` y almacenadas en formato Delta. La capa Bronze conserva los datos de origen para utilizarlos posteriormente en los procesos de limpieza y transformación de la capa Silver._

### 4.2 Capa Bronze — ingesta de datos crudos

La capa Bronze contiene una copia de los datos originales de Wanderbricks en formato Delta. En esta etapa se conserva la estructura y los tipos de datos de la fuente original, evitando realizar transformaciones importantes.

Las tablas se leen desde `samples.wanderbricks` y se almacenan en el catálogo propio `bigdata_grupo35`, dentro del esquema `wanderbricks`. Esta separación permite conservar una representación de los datos de origen y utilizarla como base para las transformaciones posteriores de la capa Silver.

Para la implementación inicial se incorporaron las tablas relacionadas con usuarios, anfitriones, propiedades, destinos, reservas, pagos, reseñas y actividad de navegación.

### 4.3 Capa plata — datos limpios y tipados

In [0]:
from pyspark.sql import functions as F

# Leer tabla Bronze
users_bronze = spark.table(
    "bigdata_grupo35.wanderbricks.bronze_users"
)

# Limpieza y preparación
users_silver = (
    users_bronze
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn(
        "company_name",
        F.when(
            F.col("company_name").isNull(),
            F.lit("No aplica")
        ).otherwise(F.trim(F.col("company_name")))
    )
)

# Guardar tabla Silver
(
    users_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "bigdata_grupo35.wanderbricks.silver_users"
    )
)

print("Tabla Silver creada correctamente: silver_users")

In [0]:
from pyspark.sql import functions as F

bookings_bronze = spark.table(
    "bigdata_grupo35.wanderbricks.bronze_bookings"
)

bookings_silver = (
    bookings_bronze
    .withColumn("status", F.lower(F.trim(F.col("status"))))
    .withColumn(
        "total_amount",
        F.col("total_amount").cast("decimal(15,2)")
    )
    .withColumn(
        "guests_count",
        F.col("guests_count").cast("int")
    )
)

(
    bookings_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "bigdata_grupo35.wanderbricks.silver_bookings"
    )
)

print("Tabla Silver creada correctamente: silver_bookings")

In [0]:
from pyspark.sql import functions as F

# Leer tabla Bronze de reseñas
reviews_bronze = spark.table(
    "bigdata_grupo35.wanderbricks.bronze_reviews"
)

# Limpieza y preparación
reviews_silver = (
    reviews_bronze
    .withColumn("comment", F.trim(F.col("comment")))
    .withColumn("rating", F.col("rating").cast("double"))
    .withColumn(
        "comment",
        F.when(
            F.col("comment").isNull(),
            F.lit("Sin comentario")
        ).otherwise(F.col("comment"))
    )
)

# Guardar tabla Silver
(
    reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "bigdata_grupo35.wanderbricks.silver_reviews"
    )
)

print("Tabla Silver creada correctamente: silver_reviews")

In [0]:
%sql
SHOW TABLES IN bigdata_grupo35.wanderbricks;

### Transformaciones de la capa Silver

En la capa Silver se aplicaron transformaciones sobre los datos de la capa Bronze para prepararlos para el análisis. En usuarios se normalizaron campos de texto y se trataron valores nulos en `company_name`. En reservas se normalizó el estado y se convirtió `total_amount` a un tipo decimal adecuado para representar valores monetarios. En reseñas se limpiaron los comentarios, se trataron los valores nulos y se tipificó la calificación como `double`.

Las tablas resultantes se almacenaron nuevamente en formato Delta dentro del esquema `bigdata_grupo35.wanderbricks`.

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
from pyspark.sql import functions as F

# Leer la tabla Bronze de clickstream
clickstream_bronze = spark.table(
    "bigdata_grupo35.wanderbricks.bronze_clickstream"
)

# Mostrar el esquema para evidenciar la estructura anidada
clickstream_bronze.printSchema()

In [0]:
clickstream_silver = (
    clickstream_bronze
    .withColumn(
        "device",
        F.col("metadata.device")
    )
    .withColumn(
        "referrer",
        F.col("metadata.referrer")
    )
)

(
    clickstream_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "bigdata_grupo35.wanderbricks.silver_clickstream"
    )
)

print("Tabla Silver creada correctamente: silver_clickstream")

In [0]:
%sql
DESCRIBE TABLE bigdata_grupo35.wanderbricks.silver_clickstream;

### Tratamiento de datos semiestructurados

La tabla `clickstream` contiene información semiestructurada en el campo `metadata`, definido como un `struct` con los atributos `device` y `referrer`. Esta estructura permite almacenar información relacionada con cada evento de navegación sin necesidad de crear una tabla independiente para cada atributo.

Para facilitar el análisis, en la capa Silver se conservaron los datos originales y se extrajeron los atributos `device` y `referrer` como columnas adicionales. De esta forma, la información anidada puede ser utilizada directamente en consultas analíticas.


### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
%sql
DESCRIBE HISTORY bigdata_grupo35.wanderbricks.silver_bookings;

### 4.5 Propiedades del Lakehouse

En esta sección se evidencian algunas propiedades de Delta Lake utilizadas en el proyecto: atomicidad, historial y evolución del esquema.

**Time Travel:** Delta Lake mantiene un historial de las operaciones realizadas sobre la tabla. En este caso se observa la versión 0 de `silver_bookings` y la operación utilizada para crearla. Este historial permite consultar versiones anteriores de los datos y conocer cuándo y cómo fueron modificados.

In [0]:
%sql
DESCRIBE DETAIL bigdata_grupo35.wanderbricks.silver_bookings;

**Atomicidad:** Las tablas Silver se almacenan en formato Delta. Las operaciones de escritura se registran como transacciones, permitiendo que los cambios se apliquen de manera consistente y evitando dejar la tabla en un estado parcialmente actualizado.

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

In [0]:
from pyspark.sql import functions as F

clickstream_silver = spark.table(
    "bigdata_grupo35.wanderbricks.silver_clickstream"
)

display(
    clickstream_silver
    .select(
        "event",
        "property_id",
        "user_id",
        "metadata",
        "device",
        "referrer"
    )
    .limit(10)
)

### 4.4 Datos semiestructurados

La tabla `silver_clickstream` contiene información semiestructurada en el campo `metadata`, almacenado como un `struct`. Esta estructura contiene atributos como `device` y `referrer`. Spark permite acceder directamente a los campos internos del `struct`, facilitando el análisis de los eventos de navegación sin necesidad de convertir toda la estructura a texto.

In [0]:
display(
    clickstream_silver
    .select(
        "event",
        "metadata.device",
        "metadata.referrer"
    )
    .limit(10)
)

In [0]:
%sql
SELECT
    event,
    metadata,
    metadata.device AS device,
    metadata.referrer AS referrer,
    property_id,
    user_id,
    timestamp
FROM bigdata_grupo35.wanderbricks.bronze_clickstream
LIMIT 10;

La tabla clickstream contiene información semiestructurada en el campo metadata, definido como un struct que contiene los atributos device y referrer. Esto permite conservar la estructura original y posteriormente acceder a sus elementos de forma individual mediante Spark SQL.

In [0]:
# Consulta 1 — pregunta que responde:
# TODO

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

In [0]:
%sql
SELECT
    property_id,
    ROUND(AVG(rating), 2) AS promedio_rating,
    COUNT(*) AS cantidad_resenas
FROM bigdata_grupo35.wanderbricks.silver_reviews
WHERE is_deleted = false
GROUP BY property_id
HAVING COUNT(*) >= 5
ORDER BY promedio_rating DESC
LIMIT 10;

In [0]:
%sql
SELECT
    property_id,
    ROUND(AVG(rating), 2) AS promedio_rating,
    COUNT(*) AS cantidad_resenas
FROM bigdata_grupo35.wanderbricks.silver_reviews
WHERE is_deleted = false
GROUP BY property_id
HAVING COUNT(*) >= 5
ORDER BY promedio_rating DESC
LIMIT 10;

In [0]:
%sql
SELECT
    property_id,
    ROUND(AVG(rating), 2) AS promedio_rating,
    COUNT(*) AS cantidad_resenas
FROM bigdata_grupo35.wanderbricks.silver_reviews
WHERE is_deleted = false
GROUP BY property_id
HAVING COUNT(*) >= 5
ORDER BY promedio_rating DESC
LIMIT 10;

In [0]:
reviews = spark.table(
    "bigdata_grupo35.wanderbricks.silver_reviews"
)

resultado_3 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .groupBy("property_id")
    .agg(
        F.round(F.avg("rating"), 2).alias("promedio_rating"),
        F.count("*").alias("cantidad_resenas")
    )
    .filter(F.col("cantidad_resenas") >= 5)
    .orderBy(F.desc("promedio_rating"))
    .limit(10)
)

display(resultado_3)

In [0]:
%sql
SELECT
    metadata.device AS dispositivo,
    COUNT(*) AS cantidad_eventos
FROM bigdata_grupo35.wanderbricks.bronze_clickstream
GROUP BY metadata.device
ORDER BY cantidad_eventos DESC;

In [0]:
clickstream = spark.table(
    "bigdata_grupo35.wanderbricks.bronze_clickstream"
)

resultado_4 = (
    clickstream
    .groupBy(F.col("metadata.device").alias("dispositivo"))
    .count()
    .withColumnRenamed("count", "cantidad_eventos")
    .orderBy(F.desc("cantidad_eventos"))
)

display(resultado_4)

In [0]:
%sql
SELECT
    event,
    COUNT(*) AS cantidad_eventos
FROM bigdata_grupo35.wanderbricks.bronze_clickstream
GROUP BY event
ORDER BY cantidad_eventos DESC;

In [0]:
resultado_5 = (
    clickstream
    .groupBy("event")
    .count()
    .withColumnRenamed("count", "cantidad_eventos")
    .orderBy(F.desc("cantidad_eventos"))
)

display(resultado_5)

Los eventos más frecuentes en el clickstream son click, filter, search y view. El evento click presenta la mayor frecuencia con 25.216 registros, seguido de filter con 24.961 y search con 24.934. Esto permite identificar las principales interacciones de los usuarios con la plataforma y puede ayudar a analizar su comportamiento de navegación.

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas